# Refitting the correction (model 2)

> **This notebook cannot run from a clone of this repository.** It needs
> per-window design matrices, 570 MB of them, derived from raw `.mff`
> recordings. Those are patient data and are not distributed. See "What you
> need before this will run" below.
>
> **Nothing here is needed to run the detector.** `../detect.ipynb` is
> self-contained. This is for refitting the correction when new donor
> recordings turn up.

## What model 2 is

The detector is two models.

* **Model 1, frozen.** The offline notebook's XGBoost *classifier* on 210
  features of the 25 ms causal chain. Never refit, never touched here.
* **Model 2, fitted here.** An XGBoost **regressor** predicting, in logit space,

  ```
  logit(teacher) − logit(frozen)
  ```

  where the *teacher* is model 1 reading the **offline** chain: 33001-tap
  zero-phase band-pass, centred 2 s SMA, whole-window z-score, 16.5 s of
  lookahead. Applied as `sigmoid(logit(frozen) + α·correction)`.

So model 2 predicts how far the causal score sits from what it would have been
with the future available, and adds that back. The offline chain only makes
training labels; it never runs at inference.

`α = 0` reproduces the frozen detector exactly and is in the grid, so the
procedure can always fall back on model 1 alone.

## The recipe, which must not drift

| | |
|---|---|
| training rows | every NREM period **≥ 25 min** of every donor recording |
| dropped | each window's own first 20 min (the normalisation seed) |
| target | `logit(teacher) − logit(frozen)` |
| objective | `reg:squarederror` |
| eta / rounds | 0.05 / 300 |
| min_child_weight / reg_lambda | 5 / 2 |
| **subsample / colsample** | **1.0 / 1.0** |
| selected | `depth` and `α`, by leave-one-**recording**-out |
| selected at | **threshold 0.70**, which never moves |

`subsample = colsample = 1.0` is not a tuning choice. It makes the booster
**deterministic**: nothing random, the seed inert, two runs byte-identical. It
also beat the 5-seed bag it replaced by +0.005 (`RESULTS.md` §10a).

**Currently shipped: depth 6, α 0.40**, fitted on 13 windows / 584 min /
77 793 rows across three recordings.

In [ ]:
"""Imports, and the recipe's constants, all from `../constants.py`.

`../detect.ipynb` imports the same file, so the two notebooks can't disagree
about DETECTION_THRESHOLD, EPOCH_SECONDS, BIN_LENGTH_MINUTES or the rest.
"""
import json
import os
import sys
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import xgboost as xgb

warnings.filterwarnings("ignore", category=UserWarning)

# The repository root, found rather than assumed: on Colab the working
# directory is /content, not the repo.  `../detect.ipynb` does the same.
def find_repo_root():
    def has_required_files(candidate):
        return (os.path.isfile(os.path.join(candidate, "constants.py"))
                and os.path.isfile(os.path.join(candidate, "models", "xgb_frozen.pkl")))

    start_dir = os.path.abspath(os.getcwd())
    candidates = [start_dir] + [os.path.abspath(os.path.join(start_dir, *[os.pardir] * levels_up))
                      for levels_up in (1, 2)]
    for base_dir in ("/content", "/content/drive/MyDrive",
                 os.path.expanduser("~"), start_dir):
        if os.path.isdir(base_dir):
            try:
                candidates += [os.path.join(base_dir, entry) for entry in sorted(os.listdir(base_dir))]
            except OSError:
                pass
    for candidate in candidates:
        if os.path.isdir(candidate) and has_required_files(candidate):
            return candidate
    raise SystemExit(
        "Could not find constants.py next to models/xgb_frozen.pkl.  This "
        "notebook has to run from inside the repository.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from constants import *      # noqa: F401,F403  - see ../constants.py
import constants as CONSTANTS

# The one setting that depends on the machine rather than on the recipe.
N_THREADS = int(os.environ.get("RETRAIN_NTHREAD", str(DEFAULT_N_THREADS)))

TREE_PARAMS = dict(tree_method="hist", max_bin=XGB_MAX_BIN, nthread=N_THREADS, seed=0)
DEPTH_GRID = [dict(depth=candidate, eta=XGB_ETA, rounds=XGB_ROUNDS) for candidate in XGB_DEPTHS]

OUT_DIR = os.path.join(REPO_ROOT, MODEL_DIR)   # where the refitted correction goes

print(f"threshold {DETECTION_THRESHOLD:.2f} (selection happens HERE and nowhere else), "
      f"seed block {NORMALISATION_MINUTES:g} min")
print(f"grid: depths {list(XGB_DEPTHS)} x {len(ALPHA_GRID)} alphas, "
      f"eta {XGB_ETA}, {XGB_ROUNDS} rounds, "
      f"subsample/colsample {XGB_SUBSAMPLE}/{XGB_COLSAMPLE} (deterministic)")

## What you need before this will run

Three things, none of them in this repository:

1. **The raw recordings.** EGI `.mff`, 1000 Hz, with E227 and E254. Patient
   data, never committed.
2. **A hypnogram per recording**, to find the NREM periods. The originals came
   from YASA over the whole night.
3. **Two design matrices per training window**, the same 210 features computed
   twice:
   * `causal`: the deployed 25 ms chain (`b25_o3 × expand20`), what model 1
     actually sees;
   * `offline`: the 16.5 s-lookahead chain, which gives the teacher's score.

   Building them takes ~45 min for 26 windows and ~570 MB of cache. In the
   research tree that is `sleep/build.py --all`.

The package that produced the shipped model is the lab's research package,
which is not distributed; §12 of `RESULTS.md` lists the commands.

### The data contract

Everything below this cell depends on one thing: a list of **window** dicts.
Produce these for your own recordings and the rest of the notebook works
unchanged.

| key | |
|---|---|
| `name` | `"SUBJECT/tag"`, e.g. `"DH3/c2"` |
| `causal_matrix` | `DataFrame`, one row per 250 ms epoch, the **210 causal** features, columns named exactly as model 1's `feature_names` |
| `frozen_proba` | model 1's probability on `causal_matrix`, the frozen score |
| `teacher_proba` | model 1's probability on the **offline** matrix, the teacher |
| `epoch_times` | epoch start times in seconds from the window start |
| `reference_times` | the teacher's detection times (stage 7 on `teacher_proba` at 0.70) |
| `n_scored_refs` | how many of those are at or after `NORMALISATION_SECONDS` |
| `duration_minutes` | window length in minutes |

In [ ]:
"""Turn a pair of (causal, offline) matrices into a training window.

`detect_batch` and `epoch_start_times` are stage 7 and the epoch clock, copied
in so this notebook does not depend on the research package.
"""
def epoch_start_times(n_epochs):
    return np.arange(n_epochs) * EPOCH_SECONDS


def detect_batch(probabilities,
                 detection_threshold=DETECTION_THRESHOLD,
                 refractory_seconds=REFRACTORY_SECONDS,
                 bin_length_minutes=BIN_LENGTH_MINUTES):
    """Stage 7: per-bin threshold crossing with a refractory skip.

    The batch form, so it drops a final bin under half a bin, which is right for
    a fixed crop.  `../detect.ipynb` runs the streaming equivalent.
    """
    epochs_per_bin = int(bin_length_minutes * EPOCHS_PER_MINUTE)
    refractory_epochs = int(refractory_seconds * EPOCHS_PER_SECOND)
    detection_times = []
    for bin_start in range(0, len(probabilities), epochs_per_bin):
        bin_probabilities = probabilities[bin_start:bin_start + epochs_per_bin]
        if len(bin_probabilities) < epochs_per_bin // 2:
            continue
        index = 0
        while index < len(bin_probabilities):
            if bin_probabilities[index] >= detection_threshold:
                detection_times.append(
                    round((bin_start + index) * EPOCH_SECONDS, TIME_DECIMALS))
                index += refractory_epochs
            else:
                index += 1
    return detection_times


def make_training_window(name, causal_matrix, frozen_proba, teacher_proba):
    """`(causal matrix, frozen probability, teacher probability) -> window`."""
    reference_times = detect_batch(teacher_proba,
                                   detection_threshold=DETECTION_THRESHOLD)
    scored_refs = sum(1 for t in reference_times if t >= NORMALISATION_SECONDS)
    return dict(
        name=name,
        causal_matrix=causal_matrix,
        frozen_proba=np.asarray(frozen_proba, float),
        teacher_proba=np.asarray(teacher_proba, float),
        epoch_times=epoch_start_times(len(causal_matrix)),
        reference_times=reference_times,
        n_scored_refs=scored_refs,
        duration_minutes=round(
            len(causal_matrix) * EPOCH_SECONDS / SECONDS_PER_MINUTE, 1))


def load_training_windows():
    """Produce the list of training windows.

    Replace this with your own loader.  In the research tree it was

        import build as B, common as C, windows as W
        xc = B.get(subject, tag, lo, hi, "causal",  build=False)
        xo = B.get(subject, tag, lo, hi, "offline", build=False)
        w  = make_training_window(f"{subject}/{tag}", xc,
                                  C.proba(xc), C.proba(xo))

    over `W.long_windows(subject)`: every NREM period of at least
    MIN_TRAINING_WINDOW_MINUTES minutes, for each donor recording.
    """
    raise NotImplementedError(
        "No design matrices are shipped with this repository.  See "
        "'What you need before this will run' above, then point this function "
        "at your own cache.")

## The correction itself

Three functions, and they are the whole of model 2.

`build_training_frame` holds the recipe's one non-obvious choice:
`drop_seed_block=True` removes every row before `NORMALISATION_SECONDS`. Those rows are a normalisation regime the detector
never runs in, and they are 44 % of the data. Dropping them is worth +0.010
(`RESULTS.md` §5).

The last column of the design matrix is `frozen_logit`: model 2 sees model 1's
own opinion, which is what lets it learn a *correction* rather than a second
detector.

In [ ]:
def logit(probability, clip_eps=PROBABILITY_CLIP_EPS):
    probability = np.clip(np.asarray(probability, float), clip_eps, 1.0 - clip_eps)
    return np.log(probability / (1.0 - probability))


def sigmoid(logit_value):
    return 1.0 / (1.0 + np.exp(-logit_value))


def fit_correction_booster(design_matrix, target, depth=XGB_DEPTHS[-1],
                           eta=XGB_ETA, rounds=XGB_ROUNDS, seed=0):
    """The shipped booster.  subsample = colsample = 1.0 -> deterministic."""
    params = dict(TREE_PARAMS, objective="reg:squarederror", eta=eta,
                  max_depth=depth, subsample=XGB_SUBSAMPLE,
                  colsample_bytree=XGB_COLSAMPLE,
                  min_child_weight=XGB_MIN_CHILD_WEIGHT,
                  reg_lambda=XGB_REG_LAMBDA, seed=int(seed))
    return xgb.train(params,
                     xgb.DMatrix(design_matrix, label=target, nthread=N_THREADS),
                     num_boost_round=rounds)


def build_training_frame(windows, drop_seed_block=True):
    """Windows -> (design matrix + frozen_logit, target)."""
    matrices, targets = [], []
    for window in windows:
        if drop_seed_block:
            scored = window["epoch_times"] >= NORMALISATION_SECONDS
        else:
            scored = slice(None)
        causal_matrix = window["causal_matrix"][scored]
        frozen_proba = window["frozen_proba"][scored]
        teacher_proba = window["teacher_proba"][scored]
        matrices.append(causal_matrix.assign(frozen_logit=logit(frozen_proba)))
        targets.append(logit(teacher_proba) - logit(frozen_proba))
    return pd.concat(matrices, ignore_index=True), np.concatenate(targets)


class CorrectionRegressor:
    """`logit(teacher) - logit(frozen)` regressor, fitted once, reused."""

    def __init__(self, windows, config, drop_seed_block=True, log=print):
        design_matrix, target = build_training_frame(windows, drop_seed_block)
        self.n_rows = len(design_matrix)
        fit_start = time.time()
        self.booster = fit_correction_booster(design_matrix, target, seed=0,
                                              **config)
        if log:
            log(f"      fit {len(design_matrix):>7d} x "
                f"{design_matrix.shape[1]} rows, depth {config['depth']}, "
                f"{time.time() - fit_start:.0f}s")
        del design_matrix, target

    def __call__(self, window):
        causal_matrix = window["causal_matrix"].assign(
            frozen_logit=logit(window["frozen_proba"]))
        return self.booster.predict(
            xgb.DMatrix(causal_matrix, nthread=N_THREADS))

## Scoring — exact epoch, and nothing looser

A detection counts only if it lands on the **same 250 ms epoch** the offline
chain fired on. Not "this epoch or the next".

Only epochs at or after `NORMALISATION_SECONDS` are scored; before that the detector emits
nothing by construction.

In [ ]:
def confusion_counts(detected_times, reference_times, from_second):
    """Exact-epoch TP / FP / FN over the span at or after `from_second`.

    Times are rounded multiples of EPOCH_SECONDS and the refractory rules out
    duplicates, so set intersection is the exact-epoch match.
    """
    detected = {round(t, TIME_DECIMALS) for t in detected_times if t >= from_second}
    reference = {round(t, TIME_DECIMALS) for t in reference_times if t >= from_second}
    true_positives = len(detected & reference)
    return dict(TP=true_positives,
                FP=len(detected) - true_positives,
                FN=len(reference) - true_positives)


def pooled_exact_epoch_f1(windows, corrections, alpha, detection_threshold):
    """Micro-averaged exact-epoch F1 over several windows at one (alpha, th)."""
    true_positives = false_positives = false_negatives = 0
    for window, correction in zip(windows, corrections):
        corrected_proba = sigmoid(
            logit(window["frozen_proba"]) + alpha * correction)
        detected_times = detect_batch(
            corrected_proba, detection_threshold=float(detection_threshold))
        window_counts = confusion_counts(detected_times,
                                         window["reference_times"],
                                         NORMALISATION_SECONDS)
        true_positives += window_counts["TP"]
        false_positives += window_counts["FP"]
        false_negatives += window_counts["FN"]
    return 2 * true_positives / max(
        2 * true_positives + false_positives + false_negatives, 1)

## Selecting `depth` and `α`

Leave-one-**recording**-out, at threshold 0.70. For each held-out recording the
correction is fitted only on the *other* recordings' windows and scored on the
held-out one's. The `(depth, α)` with the best mean pooled F1 wins.

Two knobs, 2 depths × 7 alphas, with the threshold nailed. That is on purpose:
every rule in `RESULTS.md` that gave itself more freedom did worse, which is the
signature of selection overfitting (§11).

**What this selection is not.** The `(depth, α)` here is chosen on *all* the
donor recordings, so it has no held-out patient left to validate against. **Do
not quote a score for the model this notebook saves.** The honest estimate for a
new patient is a separate, fully nested leave-one-recording-out experiment; for
the shipped model that is `RESULTS.md`'s

```
mean precision 0.388, mean recall 0.392, mean F1 0.383    (exact epoch, 0.70)
```

which is what `../README.md` reports.

In [ ]:
"""Select (depth, alpha), then fit the deployable correction on everything.

Runtime in the research tree: ~1 min once the matrices are cached.
"""
_log_lines = []


def log(*parts):
    message = " ".join(str(part) for part in parts)
    print(message, flush=True)
    _log_lines.append(message)


windows_by_subject = load_training_windows()   # {subject: [window, ...]}
SUBJECT_IDS = list(windows_by_subject)
all_windows = [window for subject in SUBJECT_IDS
               for window in windows_by_subject[subject]]

log("=== training windows ===")
for subject in SUBJECT_IDS:
    subject_windows = windows_by_subject[subject]
    log(f"  {subject}: {len(subject_windows)} windows, "
        f"{sum(w['duration_minutes'] for w in subject_windows):.1f} min, "
        f"{sum(w['n_scored_refs'] for w in subject_windows)} "
        f"scored reference spikes")

log(f"\n=== selecting depth and alpha, leave-one-recording-out, "
    f"th {DETECTION_THRESHOLD:.2f} ===")
booster_cache, scores = {}, {}
for config in DEPTH_GRID:
    for held_out in SUBJECT_IDS:
        fit_subjects = [s for s in SUBJECT_IDS if s != held_out]
        validation_windows = [w for w in windows_by_subject[held_out]
                              if w["n_scored_refs"] >= MIN_VALIDATION_REF_SPIKES]
        if not validation_windows:
            log(f"  depth {config['depth']} / hold out {held_out}: no window "
                f"with >= {MIN_VALIDATION_REF_SPIKES} spikes, skipped")
            continue
        cache_key = (tuple(fit_subjects), config["depth"])
        if cache_key not in booster_cache:
            fit_windows = [w for s in fit_subjects
                           for w in windows_by_subject[s]]
            booster_cache[cache_key] = CorrectionRegressor(
                fit_windows, config, True, log)
        corrections = [booster_cache[cache_key](w) for w in validation_windows]
        for alpha in ALPHA_GRID:
            scores.setdefault((config["depth"], alpha), []).append(
                pooled_exact_epoch_f1(validation_windows, corrections, alpha,
                                      DETECTION_THRESHOLD))
        best_alpha = max(ALPHA_GRID,
                         key=lambda a: pooled_exact_epoch_f1(
                             validation_windows, corrections, a,
                             DETECTION_THRESHOLD))
        best_f1 = pooled_exact_epoch_f1(validation_windows, corrections,
                                        best_alpha, DETECTION_THRESHOLD)
        log(f"  depth {config['depth']} / hold out {held_out}: "
            f"{len(validation_windows)} windows, "
            f"best alpha here {best_alpha:.2f} ({best_f1:.3f})")

ranked = sorted(((float(np.mean(fold_scores)), depth, alpha)
                 for (depth, alpha), fold_scores in scores.items()),
                reverse=True)
log(f"\n  {'depth':>5} {'alpha':>6} {'mean pooled F1':>15}")
for f1, depth, alpha in ranked[:8]:
    log(f"  {depth:>5} {alpha:>6.2f} {f1:>15.3f}")
inner_f1, chosen_depth, chosen_alpha = ranked[0]
chosen_config = next(c for c in DEPTH_GRID if c["depth"] == chosen_depth)
log(f"\n  chosen: depth {chosen_depth}, alpha {chosen_alpha:.2f} "
    f"(inner pooled F1 {inner_f1:.3f}) at threshold {DETECTION_THRESHOLD:.2f}")

In [ ]:
"""The final fit, on every window, and the artefact."""
# The published leave-one-recording-out result (RESULTS.md section 0), copied
# into the metadata so a deployed model carries its own honest estimate.
LOSO_PRECISION, LOSO_RECALL, LOSO_F1 = 0.388, 0.392, 0.383
log(f"\n=== final fit on all {len(all_windows)} windows ===")
design_matrix, target = build_training_frame(all_windows, drop_seed_block=True)
feature_names = list(design_matrix.columns)
assert feature_names[-1] == "frozen_logit"
fit_start = time.time()
booster = fit_correction_booster(design_matrix, target, seed=0,
                                 **chosen_config)
log(f"  {len(design_matrix)} rows x {design_matrix.shape[1]} columns, "
    f"depth {chosen_depth}, {time.time() - fit_start:.0f}s")

os.makedirs(OUT_DIR, exist_ok=True)
model_path = os.path.join(OUT_DIR, "correction.json")
booster.save_model(model_path)

# The key names here are the metadata contract `../detect.ipynb` asserts
# against on load.  Do not rename them.
metadata = dict(
    created=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    recipe="nrem_ns / hold70, RESULTS.md section 11",
    depth=chosen_depth, alpha=float(chosen_alpha),
    threshold=DETECTION_THRESHOLD, sec_limit=REFRACTORY_SECONDS,
    bin_length_min=BIN_LENGTH_MINUTES, calib_min=NORMALISATION_MINUTES,
    chain="b25_o3", norm="expand20", lookahead_ms=LOOKAHEAD_MS,
    eta=chosen_config["eta"], rounds=chosen_config["rounds"],
    subsample=XGB_SUBSAMPLE, colsample_bytree=XGB_COLSAMPLE,
    deterministic=True,
    n_rows=int(len(design_matrix)), n_features=int(design_matrix.shape[1]),
    n_windows=len(all_windows),
    train_min=round(sum(w["duration_minutes"] for w in all_windows), 1),
    windows=[w["name"] for w in all_windows],
    inner_pooled_f1=round(inner_f1, 4),
    feature_names=feature_names,
    loso_estimate=dict(
        note="leave-one-recording-out - the honest estimate of performance on "
             "a NEW patient.  This model itself has no held-out recording and "
             "must not be quoted a score.",
        # measured, RESULTS.md section 0.  Replace these if you refit on
        # different donors: they describe the shipped model, not this one.
        mean_precision=LOSO_PRECISION, mean_recall=LOSO_RECALL,
        mean_f1=LOSO_F1),
)
with open(os.path.join(OUT_DIR, "correction_meta.json"), "w") as fh:
    json.dump(metadata, fh, indent=1)
with open(os.path.join(OUT_DIR, "train_final.log"), "w") as fh:
    fh.write("\n".join(_log_lines) + "\n")

log(f"\n-> {model_path}  ({os.path.getsize(model_path) / BYTES_PER_KB:.0f} KB)")

## After refitting

`../detect.ipynb` reads `models/correction.json` and `models/correction_meta.json`
and picks up the new `alpha` automatically. Two things to check:

1. **Determinism.** Run this notebook twice. `correction.json` must be
   byte-identical. If it is not, `subsample` or `colsample` is not 1.0.
2. **The corrected count in `../detect.ipynb`'s acceptance test will change.**
   267 was this model's number on AO0. The *frozen-only* count, 291, must not
   move: model 1 is untouched, so if that changes, the problem is in the chain.

Then update the reported figures. The numbers in `../README.md` belong to the
model that shipped with it; a refit on different donors is a different model and
needs its own leave-one-recording-out estimate first.

**Do not** report the `inner pooled F1` printed above as the model's
performance. It is a *selection* statistic, computed on the recordings the model
was fitted on.